# Préannotation vidéo avec SAM 2.1

Classes : **0 connector**, **1 clip**, **2 cable**.

Ce notebook calcule les masques à partir des repères préparés sur le Mac. Le clip est mobile : dans le scénario test1 (NOK), il reste sur le connecteur ; dans test2 (OK), il est installé sur le câble. Les annotations décrivent les pièces visibles, sans attribuer automatiquement OK/NOK aux images. Les résultats sont ensuite contrôlés localement avant leur export en YOLOv8-Seg.

Dans **Exécution > Modifier le type d’exécution**, sélectionner un **GPU**. Utiliser une session dédiée à cette annotation.

## 1. Charger les repères

Sélectionner `reperes_sam2_visibilite.zip`, recréé avec le script actuel après sauvegarde des corrections. Il contient les images, les clics, les rectangles, les déclarations de visibilité et le script, sans les anciennes contraintes de clip vide. Le notebook transmet ces données à la session Colab. Relancer cette cellule pour créer un nouveau dossier de travail ; ne pas réutiliser un ancien ZIP ou les masques d’un autre essai.

In [ ]:
from pathlib import Path
import os
import stat
import subprocess
import sys
import tempfile
import zipfile
from google.colab import files

WORKDIR = Path(tempfile.mkdtemp(prefix="annotation_sam2_", dir="/content"))
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Charger une seule archive ZIP de reperes.")
name, content = next(iter(uploaded.items()))
archive_path = WORKDIR / "reperes.zip"
archive_path.write_bytes(content)

with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if len(members) > 20000 or sum(m.file_size for m in members) > 2_000_000_000:
        raise ValueError("Archive trop volumineuse pour cet essai.")
    seen = set()
    for member in members:
        relative = Path(member.filename)
        destination = (WORKDIR / relative).resolve()
        if (relative.is_absolute() or ".." in relative.parts or "\\" in member.filename
                or ":" in member.filename or not destination.is_relative_to(WORKDIR)
                or stat.S_ISLNK(member.external_attr >> 16) or destination in seen):
            raise ValueError("Chemin invalide dans l'archive.")
        seen.add(destination)
    archive.extractall(WORKDIR)

SCRIPT = WORKDIR / "scripts" / "annotate_sam2.py"
REQUIREMENTS = WORKDIR / "requirements-sam2.txt"
JOBS = sorted((WORKDIR / "jobs").glob("*/sequence.json"))
if not SCRIPT.is_file() or not REQUIREMENTS.is_file() or not JOBS:
    raise ValueError("Archive incomplete : utiliser la commande pack du projet.")
JOBS = [path.parent for path in JOBS]
print("Sequences :", ", ".join(job.name for job in JOBS))
print("Dossier :", WORKDIR)

## 2. Installer SAM 2.1

Le code SAM est fixé à une révision précise. L’extension CUDA facultative est désactivée pour éviter sa compilation ; le calcul des masques utilise bien le GPU. Les dépendances du Mac ne sont pas modifiées.

In [ ]:
install_env = dict(os.environ, SAM2_BUILD_CUDA="0")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "setuptools>=69", "wheel"],
    check=True, env=install_env
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-build-isolation", "-r", str(REQUIREMENTS)],
    check=True, env=install_env
)
subprocess.run(
    [sys.executable, "-c",
     "import torch; assert torch.cuda.is_available(), 'Activer un GPU dans Colab'; "
     "print('PyTorch :', torch.__version__); print('GPU :', torch.cuda.get_device_name(0))"],
    check=True
)

## 3. Télécharger les poids

Le modèle **SAM 2.1 Tiny** sert au premier essai. Il segmente les objets désignés par les repères ; il n’attribue pas lui-même les classes métier.

In [ ]:
import urllib.request

CHECKPOINT = WORKDIR / "sam2.1_hiera_tiny.pt"
MODEL_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt"
temporary_checkpoint = CHECKPOINT.with_suffix(".download")
if not CHECKPOINT.exists():
    urllib.request.urlretrieve(MODEL_URL, temporary_checkpoint)
    temporary_checkpoint.replace(CHECKPOINT)
print("Poids disponibles :", CHECKPOINT.name)

## 4. Propager les masques

Chaque instance est suivie séparément. Les repères peuvent se trouver sur plusieurs images ; le suivi couvre aussi les images précédant le premier repère.

Le temps de calcul dépend du GPU, du nombre d’images et du nombre d’objets. Cette étape ne constitue pas encore une validation des annotations.

In [ ]:
for job in JOBS:
    print("Sequence :", job.name, flush=True)
    subprocess.run(
        [sys.executable, str(SCRIPT), "propagate", str(job),
         "--checkpoint", str(CHECKPOINT), "--device", "cuda"],
        cwd=WORKDIR, check=True
    )
print("Calcul termine. Les masques doivent maintenant etre controles.")

## 5. Regarder quelques aperçus

Rouge : connecteur. Bleu : clip. Vert : câble. Suivre la position réelle du clip, sans le maintenir artificiellement sur la table. Vérifier que les supports et les trous de la table ne sont pas inclus dans les pièces. Les aperçus appliquent les absences déclarées avec H ; les images marquées I restent exclues de l’entraînement. SAM peut encore produire un masque brut sur une image occultée : la correction humaine l’écarte des annotations exploitées. La vue SAM brut dans review permet de le comparer. Une annotation correcte peut représenter un montage NOK.

In [ ]:
import importlib.util
import matplotlib.pyplot as plt
import cv2

spec = importlib.util.spec_from_file_location("annotation_tool", SCRIPT)
tool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tool)

for job in JOBS:
    meta, prompts = tool.load_job(job)
    run, info = tool.active_run(job, meta, prompts)
    visibility = tool.load_visibility(job, meta, prompts)
    clip_frames = sorted({int(index) for obj in prompts["objects"] if obj["class_id"] == 1 for index in obj["prompts"]})
    indices = sorted(set([0, len(meta["frames"]) - 1] + clip_frames))
    fig, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 7), squeeze=False)
    for axis, index in zip(axes[0], indices):
        frame = cv2.imread(str(tool.image_path(job, index)))
        masks = tool.checked_masks(run, info, index)
        states = tool.visibility_at(visibility, index)
        rendered = tool.overlay(frame, tool.visible_masks(masks, states), prompts["objects"])
        axis.imshow(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))
        suffix = ' / EXCLUE' if 'ignore' in states.values() else ''
        axis.set_title(f"{job.name} / image {index + 1}{suffix}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    blocked = sum(item["blocked"] for item in info["quality"].values())
    print(f"{job.name} : {blocked}/{len(meta['frames'])} images bloquees par les controles de reperes ou de geometrie.")

## 6. Télécharger le résultat

L’archive contient les séquences et leurs masques, avec les repères et les images sources. Elle ne contient aucun label d’entraînement approuvé automatiquement.

Sur le Mac, depuis le dossier du projet :

```bash
python scripts/annotate_sam2.py restore ~/Downloads/resultats_sam2_clip_mobile.zip --output data/annotations_sam2_clip_mobile_retour
python scripts/annotate_sam2.py review data/annotations_sam2_clip_mobile_retour/test1
python scripts/annotate_sam2.py review data/annotations_sam2_clip_mobile_retour/test2
```

Si le nom de téléchargement change, adapter son chemin. Utiliser un nouveau dossier de retour à chaque essai.

In [ ]:
RESULTS = WORKDIR / "resultats_sam2_clip_mobile.zip"
with zipfile.ZipFile(RESULTS, "w", zipfile.ZIP_DEFLATED) as archive:
    for job in JOBS:
        meta, prompts = tool.load_job(job)
        run, info = tool.active_run(job, meta, prompts)
        paths = [job / "sequence.json", job / "prompts.json", job / "active_run.json",
                 run / "run.json", run / "review.json"]
        if (job / "visibility.json").exists():
            paths.append(job / "visibility.json")
        paths += [tool.image_path(job, i) for i in range(len(meta["frames"]))]
        paths += [run / "masks" / f"{i:05d}.npz" for i in range(len(meta["frames"]))]
        for path in paths:
            archive.write(path, "jobs/" + job.name + "/" + path.relative_to(job).as_posix())
files.download(str(RESULTS))

## Après le contrôle

Dans `review`, **A** approuve une image, **R** l’exclut et **N/P** change d’image. **B** confirme une image entièrement vide. Toutes les pièces visibles doivent être correctement annotées avant approbation.

Pour corriger un masque, rouvrir `points` sur la séquence retournée, sélectionner le même objet avec **Tab**, ajouter les clics correctifs, puis refaire `pack` et ce notebook. Un nouveau calcul nécessite un nouveau contrôle.

Exporter les seules images approuvées :

```bash
python scripts/annotate_sam2.py export --train data/annotations_sam2_clip_mobile_retour/test1 --val data/annotations_sam2_clip_mobile_retour/test2 --output data/datasets/faisceau_seg_clip_mobile
```

L’archive `faisceau_seg_clip_mobile.zip` sert à entraîner un modèle **`yolov8n-seg.pt`** une fois toutes les instances visibles annotées. Les repères actuels ne couvrent qu’un connecteur, un clip et un câble par vidéo. Ces deux vidéos constituent un essai ; garder d’autres prises indépendantes pour mesurer la fiabilité.

Références : [SAM 2 officiel](https://github.com/facebookresearch/sam2), [format YOLO-Seg](https://docs.ultralytics.com/datasets/segment/).